# Phase 6 — Concept Drift & Evolving Fuzzy Adaptation

Dự án: **An Evolving Fuzzy Reasoning System for Sensor Stream Anomaly Detection under Concept Drift**


In [1]:
# Phase 6.0 — Setup môi trường, Static Mamdani FIS Engine và nạp 3 kịch bản Stream
from pathlib import Path
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# 1. Nạp dữ liệu gốc và trích xuất Test stream (Phase 1.5)
data_path = Path("../data/ai4i2020.csv") if Path("../data/ai4i2020.csv").exists() else Path("data/ai4i2020.csv")
df = pd.read_csv(data_path)
test_df = df.iloc[8000:].copy()
stream_df = test_df.copy().reset_index(drop=True)

# 2. Universe of Discourse (Phase 2 & Phase 4)
universes = {
    "air_temp":     np.linspace(295.3, 304.5, 1000),
    "process_temp": np.linspace(305.7, 313.8, 1000),
    "rpm":          np.linspace(1168,  2886,  1000),
    "torque":       np.linspace(3.8,   76.2,  1000),
    "tool_wear":    np.linspace(0,     253,   1000),
    "anomaly":      np.linspace(0,     1,     1000),
}

# 3. Cấu hình Static Mamdani FIS đã đóng băng (Phase 2 & Phase 3)
mf_params = {
    "air_temp": {
        "LOW":    ("trap", [295.3, 295.3, 298.0, 300.4]),
        "MEDIUM": ("tri",  [298.0, 300.4, 302.5]),
        "HIGH":   ("trap", [300.4, 302.5, 304.5, 304.5])
    },
    "process_temp": {
        "LOW":    ("trap", [305.7, 305.7, 308.0, 309.7]),
        "MEDIUM": ("tri",  [308.0, 309.7, 311.5]),
        "HIGH":   ("trap", [309.7, 311.5, 313.8, 313.8])
    },
    "rpm": {
        "LOW":    ("trap", [1168.0, 1168.0, 1350.0, 1500.0]),
        "MEDIUM": ("tri",  [1350.0, 1504.0, 1800.0]),
        "HIGH":   ("trap", [1600.0, 1880.0, 2886.0, 2886.0])
    },
    "torque": {
        "LOW":    ("trap", [3.8, 3.8, 25.0, 40.0]),
        "MEDIUM": ("tri",  [25.0, 40.0, 55.0]),
        "HIGH":   ("trap", [40.0, 55.0, 76.2, 76.2])
    },
    "tool_wear": {
        "LOW":    ("trap", [0.0, 0.0, 54.0, 109.0]),
        "MEDIUM": ("tri",  [54.0, 109.0, 164.0]),
        "HIGH":   ("trap", [109.0, 164.0, 253.0, 253.0])
    }
}

anomaly_universe = universes["anomaly"]
anomaly_mfs = {
    "LOW":    fuzz.trapmf(anomaly_universe, [0.0, 0.0, 0.25, 0.50]),
    "MEDIUM": fuzz.trimf( anomaly_universe, [0.25, 0.50, 0.75]),
    "HIGH":   fuzz.trapmf(anomaly_universe, [0.50, 0.75, 1.0, 1.0])
}

def eval_mf(x, mf_type, params):
    x = float(x)
    if mf_type == "tri":
        a, b, c = params
        if a < b and a <= x <= b:
            return (x - a) / (b - a)
        elif b < c and b <= x <= c:
            return (c - x) / (c - b)
        elif x == b:
            return 1.0
        return 0.0
    elif mf_type == "trap":
        a, b, c, d = params
        if x < a:
            return 1.0 if a == b else 0.0
        elif a <= x < b:
            return (x - a) / (b - a) if b > a else 1.0
        elif b <= x <= c:
            return 1.0
        elif c < x <= d:
            return (d - x) / (d - c) if d > c else 1.0
        else:
            return 1.0 if c == d else 0.0

def compute_memberships(sample):
    mapping = {
        "air_temp":     sample["Air temperature [K]"],
        "process_temp": sample["Process temperature [K]"],
        "rpm":          sample["Rotational speed [rpm]"],
        "torque":       sample["Torque [Nm]"],
        "tool_wear":    sample["Tool wear [min]"],
    }
    m = {}
    for var_name, val in mapping.items():
        m[var_name] = {}
        for term, (m_type, params) in mf_params[var_name].items():
            m[var_name][term] = eval_mf(val, m_type, params)
    return m

rules = [
    # HIGH anomaly — 4 rules
    ("R1", [("rpm", "LOW"), ("torque", "HIGH"), ("air_temp", "HIGH")], "HIGH"),
    ("R2", [("torque", "HIGH"), ("air_temp", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),
    ("R3", [("rpm", "LOW"), ("torque", "HIGH"), ("process_temp", "HIGH")], "HIGH"),
    ("R4", [("rpm", "LOW"), ("torque", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),

    # MEDIUM anomaly — 5 rules
    ("R5", [("rpm", "LOW"), ("torque", "HIGH")], "MEDIUM"),
    ("R6", [("rpm", "LOW"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R7", [("torque", "HIGH"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R8", [("torque", "HIGH"), ("tool_wear", "HIGH")], "MEDIUM"),
    ("R9", [("torque", "HIGH"), ("process_temp", "HIGH")], "MEDIUM"),

    # LOW anomaly — 3 rules
    ("R10", [("rpm", "MEDIUM"), ("torque", "MEDIUM")], "LOW"),
    ("R11", [("rpm", "MEDIUM"), ("torque", "LOW")], "LOW"),
    ("R12", [("rpm", "HIGH"), ("torque", "LOW")], "LOW"),
]

def mamdani_inference(sample, return_details=False):
    mu = compute_memberships(sample)
    rule_activations = {}
    implied_outputs = []
    
    for r_id, antecedents, consequent in rules:
        alpha = min(mu[var][term] for var, term in antecedents)
        rule_activations[r_id] = alpha
        implied_mf = np.fmin(alpha, anomaly_mfs[consequent])
        implied_outputs.append(implied_mf)
        
    aggregated_mf = np.zeros_like(anomaly_universe)
    for mf in implied_outputs:
        aggregated_mf = np.fmax(aggregated_mf, mf)
        
    if np.sum(aggregated_mf) == 0:
        score = 0.0
    else:
        score = fuzz.defuzz(anomaly_universe, aggregated_mf, "centroid")
        
    if return_details:
        return score, rule_activations, mu, aggregated_mf
    return score

SUDDEN_DRIFT_POINT = 1000
RPM_SHIFT = -150
TORQUE_SHIFT = 8
RPM_MIN = universes["rpm"].min()
RPM_MAX = universes["rpm"].max()
TORQUE_MIN = universes["torque"].min()
TORQUE_MAX = universes["torque"].max()

sudden_drift_stream = stream_df.copy()
after_drift = sudden_drift_stream.index >= SUDDEN_DRIFT_POINT
sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] = (
    sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] + RPM_SHIFT
).clip(RPM_MIN, RPM_MAX)
sudden_drift_stream.loc[after_drift, "Torque [Nm]"] = (
    sudden_drift_stream.loc[after_drift, "Torque [Nm]"] + TORQUE_SHIFT
).clip(TORQUE_MIN, TORQUE_MAX)

GRADUAL_DRIFT_START = 800
GRADUAL_DRIFT_END = 1200
gradual_drift_stream = stream_df.copy()
gradual_drift_stream['Rotational speed [rpm]'] = gradual_drift_stream['Rotational speed [rpm]'].astype(float)
gradual_drift_stream['Torque [Nm]'] = gradual_drift_stream['Torque [Nm]'].astype(float)
for i in gradual_drift_stream.index:
    if i < GRADUAL_DRIFT_START:
        alpha = 0.0
    elif i >= GRADUAL_DRIFT_END:
        alpha = 1.0
    else:
        alpha = (i - GRADUAL_DRIFT_START) / (GRADUAL_DRIFT_END - GRADUAL_DRIFT_START)

    gradual_drift_stream.loc[i, "Rotational speed [rpm]"] = np.clip(
        stream_df.loc[i, "Rotational speed [rpm]"] + alpha * RPM_SHIFT,
        RPM_MIN, RPM_MAX
    )
    gradual_drift_stream.loc[i, "Torque [Nm]"] = np.clip(
        stream_df.loc[i, "Torque [Nm]"] + alpha * TORQUE_SHIFT,
        TORQUE_MIN, TORQUE_MAX
    )

def score_stream(df):
    return np.array([
        mamdani_inference(row)
        for _, row in df.iterrows()
    ])

scenario_scores = {
    "Control": score_stream(stream_df),
    "Sudden Drift": score_stream(sudden_drift_stream),
    "Gradual Drift": score_stream(gradual_drift_stream)
}
print("Setup Phase 6 hoàn tất. Dữ liệu, Static FIS Engine và Anomaly Scores đã sẵn sàng.")


Setup Phase 6 hoàn tất. Dữ liệu, Static FIS Engine và Anomaly Scores đã sẵn sàng.


## Cell 6.1 — Inspect Anomaly Score Around Drift


In [2]:
import numpy as np

def summarize_score_region(scores, start, end, name):
    region = np.asarray(scores[start:end])

    print(f"\n{name}")
    print(f"Range: {start} -> {end - 1}")
    print(f"Count: {len(region)}")
    print(f"Mean: {region.mean():.4f}")
    print(f"Median: {np.median(region):.4f}")
    print(f"Std: {region.std():.4f}")
    print(f"Min: {region.min():.4f}")
    print(f"Max: {region.max():.4f}")

print("=== SUDDEN DRIFT ===")

summarize_score_region(
    scenario_scores["Sudden Drift"],
    0,
    1000,
    "Before Drift"
)

summarize_score_region(
    scenario_scores["Sudden Drift"],
    1000,
    2000,
    "After Drift"
)

print("\n=== GRADUAL DRIFT ===")

summarize_score_region(
    scenario_scores["Gradual Drift"],
    0,
    800,
    "Before Transition"
)

summarize_score_region(
    scenario_scores["Gradual Drift"],
    800,
    1200,
    "Transition"
)

summarize_score_region(
    scenario_scores["Gradual Drift"],
    1200,
    2000,
    "After Transition"
)


=== SUDDEN DRIFT ===

Before Drift
Range: 0 -> 999
Count: 1000
Mean: 0.3159
Median: 0.2249
Std: 0.1476
Min: 0.1944
Max: 0.6898

After Drift
Range: 1000 -> 1999
Count: 1000
Mean: 0.4480
Median: 0.5000
Std: 0.1739
Min: 0.1944
Max: 0.6898

=== GRADUAL DRIFT ===

Before Transition
Range: 0 -> 799
Count: 800
Mean: 0.3200
Median: 0.2251
Std: 0.1509
Min: 0.1944
Max: 0.6898

Transition
Range: 800 -> 1199
Count: 400
Mean: 0.3650
Median: 0.3191
Std: 0.1629
Min: 0.1944
Max: 0.6898

After Transition
Range: 1200 -> 1999
Count: 800
Mean: 0.4535
Median: 0.5000
Std: 0.1743
Min: 0.1944
Max: 0.6898


## Phase 6.2 — Formalize What Changed

Mục tiêu: Định lượng mức thay đổi anomaly score trước/sau drift, chưa đưa ra cơ chế thích nghi.


In [3]:
import numpy as np

print("=== PHASE 6.2 — SCORE SHIFT SUMMARY ===")

def compare_score_regions(scores, start_a, end_a, start_b, end_b, name_a, name_b):
    region_a = np.asarray(scores[start_a:end_a])
    region_b = np.asarray(scores[start_b:end_b])

    mean_a = region_a.mean()
    mean_b = region_b.mean()

    median_a = np.median(region_a)
    median_b = np.median(region_b)

    print(f"\n{name_a} vs {name_b}")
    print(f"Mean:   {mean_a:.4f} -> {mean_b:.4f}")
    print(f"Delta:  {mean_b - mean_a:+.4f}")
    print(f"Median: {median_a:.4f} -> {median_b:.4f}")
    print(f"Delta:  {median_b - median_a:+.4f}")


compare_score_regions(
    scenario_scores["Sudden Drift"],
    0, 1000,
    1000, 2000,
    "Before Drift",
    "After Drift"
)

compare_score_regions(
    scenario_scores["Gradual Drift"],
    0, 800,
    1200, 2000,
    "Before Transition",
    "After Transition"
)


=== PHASE 6.2 — SCORE SHIFT SUMMARY ===

Before Drift vs After Drift
Mean:   0.3159 -> 0.4480
Delta:  +0.1321
Median: 0.2249 -> 0.5000
Delta:  +0.2751

Before Transition vs After Transition
Mean:   0.3200 -> 0.4535
Delta:  +0.1335
Median: 0.2251 -> 0.5000
Delta:  +0.2749


## Phase 6.3 — Adaptation Trigger Check

Kiểm tra rằng các detection point của ADWIN (Phase 5) thực sự nằm sau thời điểm drift bắt đầu/kết thúc.
Nguyên tắc: Không được phép thích nghi trước detection point.


In [4]:
print("=== PHASE 6.3 — ADAPTATION TRIGGER CHECK ===")

# Known experimental boundaries
sudden_drift_start = 1000

gradual_transition_start = 800
gradual_transition_end = 1200

# ADWIN detection results from Phase 5
sudden_detections = [1183]
gradual_detections = [1247]

print("\nSudden Drift")
print(f"Injected drift starts at index: {sudden_drift_start}")
print(f"ADWIN detections: {sudden_detections}")

for detection in sudden_detections:
    print(
        f"Detection {detection}: "
        f"{'AFTER drift start' if detection >= sudden_drift_start else 'BEFORE drift start'}"
    )

print("\nGradual Drift")
print(f"Transition: {gradual_transition_start} -> {gradual_transition_end - 1}")
print(f"ADWIN detections: {gradual_detections}")

for detection in gradual_detections:
    print(
        f"Detection {detection}: "
        f"{'AFTER transition end' if detection >= gradual_transition_end else 'DURING transition'}"
    )


=== PHASE 6.3 — ADAPTATION TRIGGER CHECK ===

Sudden Drift
Injected drift starts at index: 1000
ADWIN detections: [1183]
Detection 1183: AFTER drift start

Gradual Drift
Transition: 800 -> 1199
ADWIN detections: [1247]
Detection 1247: AFTER transition end


## Phase 6.4 — Define Adaptation Data Boundary

Xác định ranh giới mẫu dữ liệu cho phép adaptation sau thời điểm ADWIN cảnh báo trôi dạt.
Bảo đảm phân tách nghiêm ngặt giữa adaptation window và evaluation region để ngăn ngừa data leakage.


In [5]:
print("=== PHASE 6.4 — ADAPTATION DATA BOUNDARY ===")

# ADWIN detection points from Phase 5
sudden_detection = 1183
gradual_detection = 1247

stream_length = len(stream_df)

print("\nSudden Drift")
print(f"Drift detection index: {sudden_detection}")
print(f"Adaptation can start from: {sudden_detection + 1}")
print(f"Remaining stream samples: {stream_length - (sudden_detection + 1)}")

print("\nGradual Drift")
print(f"Drift detection index: {gradual_detection}")
print(f"Adaptation can start from: {gradual_detection + 1}")
print(f"Remaining stream samples: {stream_length - (gradual_detection + 1)}")

print("\nRule")
print("Adaptation must NOT use samples before ADWIN detection.")
print("Samples used for adaptation must be separated from samples used for evaluation.")


=== PHASE 6.4 — ADAPTATION DATA BOUNDARY ===

Sudden Drift
Drift detection index: 1183
Adaptation can start from: 1184
Remaining stream samples: 816

Gradual Drift
Drift detection index: 1247
Adaptation can start from: 1248
Remaining stream samples: 752

Rule
Adaptation must NOT use samples before ADWIN detection.
Samples used for adaptation must be separated from samples used for evaluation.


## Phase 6.5 — Online Adaptation / Evaluation Protocol

Xác lập giao thức đánh giá trực tuyến Prequential (Test-Then-Train / Test-Then-Adapt).
Mỗi mẫu dữ liệu $x_t$ phải được đánh giá bởi trạng thái hệ mờ hiện tại trước khi được sử dụng để thích nghi.


In [6]:
print("=== PHASE 6.5 — ONLINE ADAPTATION / EVALUATION PROTOCOL ===")

sudden_detection = 1183
gradual_detection = 1247

print("\nProtocol:")
print("1. Receive sample x_t")
print("2. Compute anomaly score using current fuzzy knowledge")
print("3. Record prediction / score for evaluation")
print("4. Update drift detector")
print("5. If drift has been detected, x_t may become available for future adaptation")
print("6. Never adapt before evaluating the sample")

print("\nSudden Drift")
print(f"Detection index: {sudden_detection}")
print(f"First post-detection sample: {sudden_detection + 1}")
print(
    f"Samples available for post-detection adaptation/evaluation: "
    f"{stream_length - sudden_detection - 1}"
)

print("\nGradual Drift")
print(f"Detection index: {gradual_detection}")
print(f"First post-detection sample: {gradual_detection + 1}")
print(
    f"Samples available for post-detection adaptation/evaluation: "
    f"{stream_length - gradual_detection - 1}"
)

print("\nLeakage Check")
print("Evaluation happens BEFORE adaptation for each sample.")
print("No future sample is used to evaluate the current sample.")


=== PHASE 6.5 — ONLINE ADAPTATION / EVALUATION PROTOCOL ===

Protocol:
1. Receive sample x_t
2. Compute anomaly score using current fuzzy knowledge
3. Record prediction / score for evaluation
4. Update drift detector
5. If drift has been detected, x_t may become available for future adaptation
6. Never adapt before evaluating the sample

Sudden Drift
Detection index: 1183
First post-detection sample: 1184
Samples available for post-detection adaptation/evaluation: 816

Gradual Drift
Detection index: 1247
First post-detection sample: 1248
Samples available for post-detection adaptation/evaluation: 752

Leakage Check
Evaluation happens BEFORE adaptation for each sample.
No future sample is used to evaluate the current sample.


## Phase 6.6 — Inspect Fuzzy Knowledge Candidates for Adaptation

Xác định các thành phần tri thức mờ được phép thích nghi (Candidate Adaptation Targets) và các tri thức được bảo vệ (Protected Knowledge).


In [7]:
print("=== PHASE 6.6 — FUZZY KNOWLEDGE CANDIDATES ===")

print("\nStatic Fuzzy Knowledge")
print("Inputs:")
print("- Air temperature")
print("- Process temperature")
print("- RPM")
print("- Torque")
print("- Tool wear")

print("\nControlled Drift Variables:")
print("- RPM")
print("- Torque")

print("\nNon-Drift Variables:")
print("- Air temperature")
print("- Process temperature")
print("- Tool wear")

print("\nCandidate Adaptation Targets:")
print("1. RPM membership functions")
print("2. Torque membership functions")

print("\nProtected Knowledge:")
print("Air temperature membership functions")
print("Process temperature membership functions")
print("Tool wear membership functions")
print("Static rule structure")
print("Static validation threshold")


=== PHASE 6.6 — FUZZY KNOWLEDGE CANDIDATES ===

Static Fuzzy Knowledge
Inputs:
- Air temperature
- Process temperature
- RPM
- Torque
- Tool wear

Controlled Drift Variables:
- RPM
- Torque

Non-Drift Variables:
- Air temperature
- Process temperature
- Tool wear

Candidate Adaptation Targets:
1. RPM membership functions
2. Torque membership functions

Protected Knowledge:
Air temperature membership functions
Process temperature membership functions
Tool wear membership functions
Static rule structure
Static validation threshold
